# Topic 1: Join Types in Spark

> **Phase 3 → Week 4 → Topic 1**
>
> Prerequisites: Week 3 complete (DataFrames Part 1)

---

## The Restaurant Bill Analogy

Imagine two lists at a restaurant:
- **Table A**: List of guests who ordered food
- **Table B**: List of guests who ordered drinks

Different join types answer different questions:
- **INNER JOIN** → Who ordered BOTH food AND drinks?
- **LEFT JOIN** → All food orders, and drinks if they exist (food guests without drinks = null drink)
- **RIGHT JOIN** → All drink orders, and food if it exists
- **FULL OUTER JOIN** → Everyone — food-only, drinks-only, and both
- **LEFT SEMI JOIN** → Which food guests also ordered drinks? (only show food side)
- **LEFT ANTI JOIN** → Which food guests did NOT order drinks?
- **CROSS JOIN** → Every food item paired with every drink (dangerous — huge result)

---

## Spark Join Internals — 3 Physical Strategies

When you call `.join()`, Spark picks ONE of three physical strategies:

### 1. Broadcast Hash Join (BHJ) — Fastest
```
Small table → broadcast to ALL executors (fits in memory)
Large table → each partition looks up the small table locally
NO SHUFFLE needed

Used when: one table < spark.sql.autoBroadcastJoinThreshold (default 10MB)
```

### 2. Sort Merge Join (SMJ) — Default for large tables
```
Both tables → shuffle by join key → sort → merge
Shuffle is expensive (data moves across network)

Used when: both tables too large to broadcast
```

### 3. Shuffle Hash Join
```
One side shuffled into hash buckets
Other side probes the hash table
Used in specific cases by AQE
```

**Interview rule**: For join optimization, always ask "can I broadcast the smaller table?"

---

## The Data Skew Problem

```
Normal distribution:           Skewed distribution:
Partition 1: 1000 rows         Partition 1:    50 rows
Partition 2: 1000 rows    vs   Partition 2:    50 rows
Partition 3: 1000 rows         Partition 3: 99900 rows  ← hotspot!

One executor does 99% of the work → job stalls on that one task
```

Fix: **Salting** — add a random prefix to the skewed key to spread it across partitions.

---

## Interview Cheat Sheet

**Q: What is a broadcast join and when should you use it?**
> A broadcast join copies the smaller table to every executor, so each executor can perform the join locally without shuffling data across the network. Use it when one table is small enough to fit in memory (under `spark.sql.autoBroadcastJoinThreshold`, default 10MB). It eliminates the most expensive part of a join — the shuffle.

**Q: What is data skew in joins, and how do you fix it?**
> Data skew happens when most rows share the same join key, causing one partition to receive the vast majority of data. One executor becomes the bottleneck. Fix: salting — add a random integer prefix (0 to N) to the skewed key to spread it across N partitions, then explode the other side of the join to have all N versions of each key.

**Q: What's the difference between LEFT SEMI and INNER join?**
> Both return only rows that have a match, but LEFT SEMI only returns columns from the LEFT table — it never duplicates rows when the right side has multiple matches. It's equivalent to `WHERE EXISTS (SELECT 1 FROM right WHERE right.key = left.key)` in SQL.

**Q: What's LEFT ANTI join used for?**
> LEFT ANTI returns rows from the LEFT table that have NO match in the right table. Classic use case: finding customers who have never placed an order, or products never sold.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("Week4 - Join Types") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.autoBroadcastJoinThreshold", "-1") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} ready")

In [ ]:
# Load our datasets
customers = spark.read.csv("/workspace/week4/data/customers.csv", header=True, inferSchema=True)
orders    = spark.read.csv("/workspace/week4/data/orders.csv",    header=True, inferSchema=True)
products  = spark.read.csv("/workspace/week4/data/products.csv",  header=True, inferSchema=True)

print(f"customers: {customers.count()} rows")
customers.show()
print(f"orders: {orders.count()} rows")
orders.show()
print(f"products: {products.count()} rows")
products.show()

## Part 1: The 7 Join Types

In [ ]:
# INNER JOIN — only rows that match in BOTH tables
# orders has customer_id 11 (no matching customer) — it disappears
inner = orders.join(customers, on="customer_id", how="inner")
print(f"INNER JOIN: orders({orders.count()}) x customers({customers.count()}) = {inner.count()} rows")
print("Note: order O016 (customer_id=11) dropped — no matching customer")
inner.select("order_id", "customer_id", "name", "amount").show()

In [ ]:
# LEFT JOIN (LEFT OUTER) — all rows from LEFT (orders), match from RIGHT if exists
# O016 with customer_id=11 is kept, customer columns are null
left = orders.join(customers, on="customer_id", how="left")
print(f"LEFT JOIN: {left.count()} rows (all orders kept)")
# Show the unmatched row
print("Unmatched order (customer not in customers table):")
left.filter(F.col("name").isNull()).select("order_id", "customer_id", "name", "city").show()

In [ ]:
# RIGHT JOIN (RIGHT OUTER) — all rows from RIGHT (customers), match from LEFT if exists
# Customers who never ordered appear with null order columns
right = orders.join(customers, on="customer_id", how="right")
print(f"RIGHT JOIN: {right.count()} rows (all customers kept)")
print("Customers with no orders:")
right.filter(F.col("order_id").isNull()).select("customer_id", "name", "country").show()

In [ ]:
# FULL OUTER JOIN — ALL rows from BOTH sides, nulls where no match
full = orders.join(customers, on="customer_id", how="full")
print(f"FULL OUTER JOIN: {full.count()} rows")
print("Unmatched from BOTH sides:")
full.filter(F.col("order_id").isNull() | F.col("name").isNull()) \
    .select("order_id", "customer_id", "name").show()

In [ ]:
# LEFT SEMI JOIN — rows from LEFT where a match EXISTS in RIGHT
# Only LEFT columns returned — no column duplication!
semi = customers.join(orders, on="customer_id", how="left_semi")
print(f"LEFT SEMI JOIN: {semi.count()} customers who have placed at least 1 order")
print("Only customer columns returned (no order columns):")
semi.show()

# Compare: INNER JOIN would duplicate a customer who placed 3 orders into 3 rows
inner_count = customers.join(orders, on="customer_id", how="inner").count()
print(f"INNER would give {inner_count} rows (duplicates per order); SEMI gives {semi.count()} (one per customer)")

In [ ]:
# LEFT ANTI JOIN — rows from LEFT where NO match in RIGHT
# "Customers who have NEVER placed an order"
anti = customers.join(orders, on="customer_id", how="left_anti")
print(f"LEFT ANTI JOIN: {anti.count()} customers with no orders")
anti.show()

# Another use case: find products never ordered
unordered = products.join(orders, on="product_id", how="left_anti")
print(f"Products never ordered:")
unordered.select("product_id", "name", "category").show()

In [ ]:
# CROSS JOIN — cartesian product (every row x every row)
# VERY dangerous on large tables: 1M x 1M = 1 TRILLION rows
# Requires explicit .crossJoin() or how="cross"

small_a = spark.createDataFrame([(1, "A"), (2, "B")],     ["id", "val_a"])
small_b = spark.createDataFrame([(10, "X"), (20, "Y")],   ["id", "val_b"])

cross = small_a.crossJoin(small_b)
print(f"CROSS JOIN: {small_a.count()} x {small_b.count()} = {cross.count()} rows")
cross.show()

print("WARNING: Never use crossJoin on production tables without a WHERE filter!")

## Part 2: Multi-Table Joins

In [ ]:
# Chain multiple joins: orders + customers + products
# Real-world: enrich an orders table with customer and product details
enriched = (
    orders
    .join(customers, on="customer_id", how="inner")
    .join(products,  on="product_id",  how="inner")
    .select(
        "order_id",
        customers["name"].alias("customer_name"),
        "country",
        products["name"].alias("product_name"),
        "category",
        "quantity",
        "amount",
        "status",
        "order_date"
    )
)

print("Enriched orders (customers + products joined):")
enriched.show(truncate=False)

In [ ]:
# Ambiguous column names — handle with aliases
# Both orders and products have a 'name' column → must disambiguate

# Method 1: rename before join
orders_renamed   = orders.withColumnRenamed("status", "order_status")
products_renamed = products.withColumnRenamed("name", "product_name")

# Method 2: use table aliases and select with dot notation
o = orders.alias("o")
p = products.alias("p")

result = o.join(p, on="product_id", how="inner") \
           .select(
               F.col("o.order_id"),
               F.col("p.name").alias("product_name"),
               F.col("p.category"),
               F.col("o.amount")
           )
result.show()

## Part 3: Broadcast Join — Performance Optimization

In [ ]:
# Force a broadcast join with the broadcast() hint
# Use when one side is small (dimension table, lookup table)

from pyspark.sql.functions import broadcast

# products is small (10 rows) — perfect broadcast candidate
result_broadcast = orders.join(
    broadcast(products),   # hint: broadcast this table
    on="product_id",
    how="inner"
)

print("With broadcast hint — check the physical plan for 'BroadcastHashJoin':")
result_broadcast.explain()

# vs without broadcast
print("\nWithout broadcast hint — check for 'SortMergeJoin':")
orders.join(products, on="product_id", how="inner").explain()

In [ ]:
# Auto-broadcast threshold (default 10MB)
print(f"Current broadcast threshold: {spark.conf.get('spark.sql.autoBroadcastJoinThreshold')} bytes")

# Re-enable auto broadcast and see Catalyst auto-detect small tables
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))  # 10MB

auto_broadcast = orders.join(products, on="product_id", how="inner")
print("\nWith auto broadcast threshold = 10MB:")
auto_broadcast.explain()

## Part 4: Join on Multiple Columns & Non-Equi Joins

In [ ]:
# Multi-column join
df1 = spark.createDataFrame([
    (1, "A", 100), (1, "B", 200), (2, "A", 300)
], ["id", "type", "value1"])

df2 = spark.createDataFrame([
    (1, "A", 10), (1, "C", 20), (2, "A", 30)
], ["id", "type", "value2"])

# Join on BOTH id AND type
multi_join = df1.join(df2, on=["id", "type"], how="inner")
print("Multi-column join on [id, type]:")
multi_join.show()

# Non-equi join (range join) — join on a CONDITION not equality
# Note: non-equi joins cannot use broadcast hash join, always SortMergeJoin
range_join = df1.join(df2, on=(df1.id == df2.id) & (df1.value1 > df2.value2), how="inner")
print("Non-equi join (value1 > value2):")
range_join.show()

## Part 5: Data Skew & Salting

In [ ]:
import random

# Simulate a skewed dataset — most rows have key='A'
skewed_data = [("A", i) for i in range(10000)] + \
              [("B", i) for i in range(50)]    + \
              [("C", i) for i in range(50)]

skewed_df = spark.createDataFrame(skewed_data, ["key", "value"])
print("Skewed data distribution:")
skewed_df.groupBy("key").count().show()

lookup_df = spark.createDataFrame([("A", "Alpha"), ("B", "Beta"), ("C", "Gamma")], ["key", "label"])

In [ ]:
# SALTING FIX for data skew
# Step 1: Add a random salt (0 to N-1) to the skewed side
# Step 2: Explode the lookup side to have all N salted versions

SALT_FACTOR = 4  # spread key='A' across 4 partitions

# Salt the skewed side
salted_skewed = skewed_df.withColumn(
    "salted_key",
    F.concat(F.col("key"), F.lit("_"), (F.rand() * SALT_FACTOR).cast("int").cast("string"))
)

# Explode the lookup side (replicate each key SALT_FACTOR times)
# Rename key to lkey to avoid ambiguity after join
salt_values = spark.range(SALT_FACTOR).select(F.col("id").alias("salt"))
salted_lookup = lookup_df.withColumnRenamed("key", "lkey").crossJoin(salt_values).withColumn(
    "salted_key",
    F.concat(F.col("lkey"), F.lit("_"), F.col("salt").cast("string"))
)

# Join on the salted key, then drop temp columns
result = salted_skewed.join(broadcast(salted_lookup), on="salted_key", how="inner") \
                      .drop("salted_key", "salt", "lkey")

print(f"Result count: {result.count()} (same as original {skewed_df.count()})")
result.groupBy("key", "label").count().show()


## Exercises

1. Find all orders with their customer name and country, but only for customers from `India` or `USA`. Use a join + filter.
2. Find products that have never been ordered. Use LEFT ANTI join.
3. Find customers who have more than 2 orders. Use a join + groupBy + filter. (Hint: semi join can help here too.)
4. Join orders with products to find the total revenue per category. Show results sorted by revenue descending.
5. Explain (in your own words) why `LEFT SEMI` is better than `INNER JOIN` when you only care about which customers placed orders.

In [ ]:
# Exercise 1: Orders for India/USA customers
india_usa = customers.filter(F.col("country").isin("India", "USA"))
orders.join(india_usa, on="customer_id", how="inner") \
      .select("order_id", "name", "country", "amount", "status") \
      .show()

In [ ]:
# Exercise 4: Revenue per category
orders.join(products, on="product_id", how="inner") \
      .groupBy("category") \
      .agg(F.round(F.sum("amount"), 2).alias("total_revenue"),
           F.count("order_id").alias("total_orders")) \
      .orderBy(F.col("total_revenue").desc()) \
      .show()